# Predictive Analytics: Support Vector Machine (SVM)

This notebook implements a complete machine learning pipeline using **Support Vector Regression (SVR)** to predict taxi trip demand in Chicago under various spatio-temporal resolutions.

In [11]:
import warnings
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score)

import joblib
from pathlib import Path

warnings.filterwarnings("ignore", category=UserWarning)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

MODELS_DIR = Path("..") / "models" # Directory to save trained models
RESOLUTION = 8  # Change to 8 if using resolution 8 data
TARGET = "Total_Trip_Start" # Target variable for prediction

In [12]:
# Select features
BASIC_FEATURES = [
    "hour_sin", "hour_cos", "month_sin", "month_cos",
    "is_weekend", "is_holiday", "is_near_holiday", "day_of_week",
    "lat", "lon", "scikit_distance_to_loop", "base_demand"
]
POI_COLS = ['poi_cat_automotive','poi_cat_civic_community', 'poi_cat_education', 'poi_cat_entertainment',
            'poi_cat_finance', 'poi_cat_food_drink', 'poi_cat_grocery', 'poi_cat_health', 'poi_cat_leisure_sports', 
            'poi_cat_lodging','poi_cat_nightlife', 'poi_cat_services', 'poi_cat_shopping','poi_cat_transport']
WEATHER = ["2m_temp_c", "total_precip_mm", "snow_cov", "snow_depth", "wind_speed"]

# Different feature sets to evaluate
FEATURE_SETS = {
    "basic": BASIC_FEATURES,
    "basic+poi": BASIC_FEATURES + POI_COLS,
    "basic+weather": BASIC_FEATURES + WEATHER,
    "basic+poi+weather": BASIC_FEATURES + POI_COLS + WEATHER,
}

# print overview of feature sets and number of features
for name, features in FEATURE_SETS.items():
    print(f"{name}: {len(features)} features ({len(POI_COLS)} POI categories)")

basic: 12 features (14 POI categories)
basic+poi: 26 features (14 POI categories)
basic+weather: 17 features (14 POI categories)
basic+poi+weather: 31 features (14 POI categories)


# Function to simplify the training Loop
This function will be called in the training loop later, where models are trained on different Datasets

### Load the temporal data splits

In [13]:
def load_datasplits():
      """
      Load the train, validation, and test datasets for the resolution set in the
      global RESOLUTION (7 or 8).
      """
      # Indicate if resolution 7 or 8 is being used
      if RESOLUTION == 7:
            print("Using Resolution 7 data.")
            # Import the data-splits for Resolution 7
            df_train = pd.read_parquet("../data/prediction_split/res_7/df_train.parquet")
            df_val = pd.read_parquet("../data/prediction_split/res_7/df_val.parquet")
            df_test = pd.read_parquet("../data/prediction_split/res_7/df_test.parquet")
      elif RESOLUTION == 8:
            print("Using Resolution 8 data.")
            # Import the data-splits for Resolution 8
            df_train = pd.read_parquet("../data/prediction_split/res_8/df_train.parquet")
            df_val = pd.read_parquet("../data/prediction_split/res_8/df_val.parquet")
            df_test = pd.read_parquet("../data/prediction_split/res_8/df_test.parquet")
      print(f"Train split loaded. Shape: {df_train.shape}")
      print(f"Validation split loaded. Shape: {df_val.shape}")
      print(f"Test split loaded. Shape: {df_test.shape}")

      zero_frac = (df_train[TARGET] == 0).mean()
      print(f"\nTarget '{TARGET}': mean={df_train[TARGET].mean():.3f}, "
            f"median={df_train[TARGET].median():.0f}, max={df_train[TARGET].max():.0f}")
      print(f"{zero_frac:.1%} of the training data have zero demand.")
      print(f"When sampling uniformly, we would draw ~{zero_frac:.0%} zeros." )
      print(f"Instead we will draw a balanced sample, all non-zero rows plus an equal number of zeros.")

      return df_train, df_val, df_test

### Input & Output Transformation

In [14]:
# Fit the data scaler on the training data and transform train, val, test sets
def transform_data(df_train, df_val, df_test, features):
    """Transform features and target for train, val, test sets."""
    # Fitting the scaler on the training data to avoid data leakage
    scaler = StandardScaler().fit(df_train[features])
    
    # Transform features and target for train, val, test sets
    X_train = scaler.transform(df_train[features])
    y_train = df_train[TARGET].values
    
    X_val = scaler.transform(df_val[features])
    y_val = df_val[TARGET].values
    
    X_test = scaler.transform(df_test[features])
    y_test = df_test[TARGET].values
    
    return X_train, y_train, X_val, y_val, X_test, y_test, scaler

def to_log1p(y):
    """Apply log1p transformation to the target variable."""
    return np.log1p(y)

# Invert log1p and clip negative demand to 0
def to_counts(pred_log):
    """Invert log1p and clip negative demand to 0."""
    return np.clip(np.expm1(pred_log), 0, None)

# Score the model performance
def score(y_true, pred_counts, name):
    mae = mean_absolute_error(y_true, pred_counts)
    rmse = np.sqrt(mean_squared_error(y_true, pred_counts))
    r2 = r2_score(y_true, pred_counts)
    print(f"{name:28s} MAE={mae:6.3f}  RMSE={rmse:6.3f}  R2={r2:7.4f}")
    return {"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2}

# Function to create a balanced sample of the training data for model training
# Extract all non-zero demand rows and sample an equal number of zero-demand rows to create a balanced dataset.
def balanced_sample(df, n_per_class=12000, seed=RANDOM_STATE):
    nz = df[df[TARGET] > 0]
    z = df[df[TARGET] == 0]
    nz_s = nz.sample(n=min(n_per_class, len(nz)), random_state=seed)
    z_s = z.sample(n=min(n_per_class, len(z)), random_state=seed)
    return pd.concat([nz_s, z_s]).sample(frac=1, random_state=seed)

### Kernel comparison

In [15]:
# Train SVR models starting with a Linear kernel and progressively introduce Polynomial and RBF kernels.
def find_best_kernel(Xs, ys, X_val, y_val):
    """
    Train SVR models with different kernels and evaluate their performance on the validation set.
    
    Parameters:
        Xs (array-like): Features of the balanced training sample.
        ys (array-like): Target values of the balanced training sample.
        X_val (array-like): Features of the validation set.
        y_val (array-like): True target values of the validation set.
    
    Returns:
        best_kernel (str): The kernel with the highest R2 score on the validation set.
        val_r2 (dict): Dictionary containing R2 scores for each kernel.
    """
    KERNELS = ["linear", "poly", "rbf"] # List of kernels to try for SVR
    val_r2 = {}

    for kernel in KERNELS:
        print(f"Training SVR with {kernel} kernel...")
        svr = SVR(kernel=kernel, C=1.0, gamma="scale", cache_size=500)
        svr.fit(Xs, ys) # Fit on balanced sample to speed up training (SVR scales poorly with large datasets)
        y_pred = to_counts(svr.predict(X_val))
        val_r2[kernel] = r2_score(y_val, y_pred)
        print(f"SVR({kernel:6s}) validation R2 = {val_r2[kernel]:.4f}")

    best_kernel = max(val_r2, key=val_r2.get)
    print(f"\nBest kernel on validation: {best_kernel}")
    return best_kernel, val_r2

### Hyperparameter Grid Search (CV on the training sample)

In [16]:
# Optimize the `C` and `gamma` hyperparameters for best kernel via grid search.
def grid_search_svr(Xs, ys, best_kernel):
    # Note: hyperparameters are selected via 3-fold CV on the balanced training
    # sample (Xs, ys); the validation set is not used here.
    print(f"\nPerforming GridSearchCV for SVR with {best_kernel} kernel...")
    # Parameter to optimize for SVR using GridSearchCV
    param_grid = {
        "C": [1, 10, 50], # Regularization parameter
        "gamma": ["scale", 0.05, 0.2], # Kernel coefficient for 'rbf' and 'poly'
    }
    if best_kernel == "poly":
        param_grid["degree"] = [2, 3]

    grid = GridSearchCV(
        SVR(kernel=best_kernel, cache_size=500),
        param_grid, cv=3, scoring="neg_mean_absolute_error", n_jobs=-1,
    )
    grid.fit(Xs, ys)
    print("Best params:", grid.best_params_)

    tuned = grid.best_estimator_
    return tuned, grid.best_params_

### Save the best model

In [17]:
# Save the tuned SVR to `../models/` so it can be reused without retraining. 
def save_model_bundle(tuned, scaler, features, feature_set, best_kernel, best_params, df_metrics):
    """
    Save the best model bundle to disk.
    
    Parameters:
        tuned: The fitted SVR model (best_estimator_ from grid search).
        scaler: The fitted StandardScaler.
        features: List of feature column names expected by the scaler.
        feature_set: Name of the feature set used.
        best_kernel: The kernel used in the best SVR model.
        best_params: The best parameters found during grid search.
        df_metrics: DataFrame containing the performance metrics of the models.
    """
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    # Bundle the estimator with everything needed to make predictions later.
    best_metrics = df_metrics.iloc[0].to_dict()
    model_bundle = {
        "model": tuned,                 # fitted SVR (best_estimator_ from grid search)
        "scaler": scaler,               # fitted StandardScaler
        "features": features,           # feature column order expected by the scaler
        "feature_set": feature_set,     # name of the feature set used
        "target": TARGET,
        "kernel": best_kernel,
        "best_params": best_params,
        "transform": "log1p",           # target was trained on log1p(y); invert with expm1 + clip>=0
        "metrics": best_metrics,        # validation-set metrics (MAE/RMSE/R2)
    }

    # Naming convention: svr_best_<res>_<feature_set>.joblib
    model_path = MODELS_DIR / f"svr_best_{RESOLUTION}_{feature_set}.joblib"
    joblib.dump(model_bundle, model_path)
    print(f"Saved best model -> {model_path.resolve()}")
    print(f"  kernel={best_kernel}, params={best_params}")
    print(f"  validation metrics: MAE={best_metrics['MAE']:.3f}  "
        f"RMSE={best_metrics['RMSE']:.3f}  R2={best_metrics['R2']:.4f}")

In [18]:
#| label: svr_final_training_loop
# Final training loop over all feature sets for the model selection
print(f"Training loop on data with resolution {RESOLUTION}")
# Load the data splits for the specified resolution
df_train, df_val, df_test = load_datasplits()
print(40*"-")
for FEATURE_SET, features in FEATURE_SETS.items():
    print(f"\nTraining SVR model for feature set: {FEATURE_SET} ({len(features)} features)")

    # Transform the data using the selected features (only the validation arrays)
    _, _, X_val, y_val, _, _, scaler = transform_data(df_train, df_val, df_test, features)

    # Create a balanced sample of the training data
    sample = balanced_sample(df_train)
    Xs = scaler.transform(sample[features])
    ys = to_log1p(sample[TARGET].values)
    print(f"Balanced training sample: {len(sample)} rows with {(sample[TARGET] > 0).mean():.0%} non-zero.")

    # Train and evaluate SVR models with different kernels
    best_kernel, _ = find_best_kernel(Xs, ys, X_val, y_val)

    # Parameter tuning using GridSearchCV (3-fold CV on the balanced sample)
    tuned, best_params = grid_search_svr(Xs, ys, best_kernel)

    # Evaluate the tuned model on the validation set. The test set is deliberately
    # held out during model/feature-set selection to avoid test-set leakage; it
    # should only be touched once, for the finally chosen model.
    val_pred = to_counts(tuned.predict(X_val))
    df_metrics = pd.DataFrame([score(y_val, val_pred, f"Tuned SVR ({best_kernel})")])

    save_model_bundle(tuned, scaler, features, FEATURE_SET, best_kernel, best_params, df_metrics)
    print(40*"_")

Training loop on data with resolution 8
Using Resolution 8 data.
Train split loaded. Shape: (4641696, 55)
Validation split loaded. Shape: (1852320, 55)
Test split loaded. Shape: (2778934, 55)

Target 'Total_Trip_Start': mean=0.585, median=0, max=328
97.1% of the training data have zero demand.
When sampling uniformly, we would draw ~97% zeros.
Instead we will draw a balanced sample, all non-zero rows plus an equal number of zeros.
----------------------------------------

Training SVR model for feature set: basic (12 features)
Balanced training sample: 24000 rows with 50% non-zero.
Training SVR with linear kernel...
SVR(linear) validation R2 = -5537.6327
Training SVR with poly kernel...
SVR(poly  ) validation R2 = 0.8306
Training SVR with rbf kernel...
SVR(rbf   ) validation R2 = 0.8842

Best kernel on validation: rbf

Performing GridSearchCV for SVR with rbf kernel...
Best params: {'C': 50, 'gamma': 0.05}
Tuned SVR (rbf)              MAE= 0.324  RMSE= 2.425  R2= 0.8874
Saved best mode

In [19]:
# 1. Pick the winning feature set by R2
best_model = ""
for model in MODELS_DIR.glob(f"svr_best_{RESOLUTION}_*.joblib"):
    b = joblib.load(model)
    if best_model == "" or b["metrics"]["R2"] > best_model["metrics"]["R2"]:
        best_model = b
# Rename best model file to indicate it is the final model for this resolution
best_model_path = MODELS_DIR / f"svr_final_{RESOLUTION}_{best_model['feature_set']}.joblib"
joblib.dump(best_model, best_model_path)
print(f"\nBest model selected: {best_model_path.resolve()}")

# Evaluate the chosen model on the test set
scaler, tuned, features = best_model["scaler"], best_model["model"], best_model["features"]
X_test = scaler.transform(df_test[features])
y_test = df_test[TARGET].values
test_pred = to_counts(tuned.predict(X_test))
name = f"pred_svr_{RESOLUTION}_{best_model['feature_set']}"
test_metrics = score(y_test, test_pred, name)

pd.DataFrame({"y_true": y_test, "y_pred": test_pred}).to_csv(MODELS_DIR / f"predictions/{name}.csv", index=False)


Best model selected: /Users/tjorgeorlitz/Documents/Uni/Semester 2/AAA/Project/AAA_Project_Group7/models/svr_final_8_basic+poi.joblib
pred_svr_8_basic+poi         MAE= 0.345  RMSE= 2.500  R2= 0.8628


In [20]:
# TODO: Add Feature Importance Analysis (SHAP or similar)
# TODO: To push further: a coarser H3 resolution (fewer all-zero cells), 
# or quantile/Tweedie losses, would suit the zero-inflated target.

# Export predictions for the visualization notebook
#np.save("../models/svm_test_preds.npy", y_pred_best)
#print("SVM test predictions saved.")


"""
def evaluate_model(tuned, X_test, y_test, best_kernel):

    Evaluate the tuned SVR model on the test set and return the performance metrics.
    
    Parameters:
        tuned: The tuned SVR model.
        X_test (array-like): Features of the test set.
        y_test (array-like): True target values of the test set.
        best_kernel (str): The kernel used in the tuned SVR model.

    Returns:
        list: A list of performance metrics for the evaluated model.

    results = []
    results.append(score(y_test, to_counts(tuned.predict(X_test)), f"Tuned SVR ({best_kernel})"))

    df_metrics = pd.DataFrame(results)
    print("\nModel Performance Comparison on the Test set)")
    print(df_metrics.to_string(index=False))

    # Pick the best model by test R2 for the spatial analysis below
    best_row = df_metrics.loc[df_metrics["R2"].idxmax(), "Model"]
    y_pred_best = {
        f"Tuned SVR ({best_kernel})": to_counts(tuned.predict(X_test)),
    }[best_row]
    print(f"\nBest model for spatial analysis: {best_row}")

    """

'\ndef evaluate_model(tuned, X_test, y_test, best_kernel):\n\n    Evaluate the tuned SVR model on the test set and return the performance metrics.\n\n    Parameters:\n        tuned: The tuned SVR model.\n        X_test (array-like): Features of the test set.\n        y_test (array-like): True target values of the test set.\n        best_kernel (str): The kernel used in the tuned SVR model.\n\n    Returns:\n        list: A list of performance metrics for the evaluated model.\n\n    results = []\n    results.append(score(y_test, to_counts(tuned.predict(X_test)), f"Tuned SVR ({best_kernel})"))\n\n    df_metrics = pd.DataFrame(results)\n    print("\nModel Performance Comparison on the Test set)")\n    print(df_metrics.to_string(index=False))\n\n    # Pick the best model by test R2 for the spatial analysis below\n    best_row = df_metrics.loc[df_metrics["R2"].idxmax(), "Model"]\n    y_pred_best = {\n        f"Tuned SVR ({best_kernel})": to_counts(tuned.predict(X_test)),\n    }[best_row]\n  